# Visualize Subgraph Representation Adjacency and

**Goal**: By viewing the Adj, to see how graphs are connected.

**Tasks:**

- Visualize one document sample (Adjacency heatmap and pyvis.
- Calculate the average adjacency matrix heatmap

In [1]:
import networkx as nx
import torch
import random
import glob
import seaborn as sns

from torch_geometric import edge_index
from torch_geometric.utils import from_networkx
from torch_geometric.utils import to_dense_adj
from matplotlib import pyplot as plt
from wasabi import color
import tqdm

DATASET = "IMDB"
GRAPHS_PATH = f"../data/datasets/{DATASET}/train/interim"
GRAPHS = glob.glob(GRAPHS_PATH + "/*.pt")
MAX_NODES = 500

random.seed(42)
random.shuffle(GRAPHS)


In [2]:
graph_data_path = GRAPHS[0]
doc_name, label, graph_nx = torch.load(graph_data_path)

/tmp/ipykernel_186731/1016278556.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  doc_name, label, graph_nx = torch.load(graph_data_path)


In [3]:
print(f"Document name:   {doc_name}")
print(f"Label:           {label}")


Document name:   19803.txt
Label:           negative


In [4]:
graph_pyg = from_networkx(graph_nx, group_edge_attrs=['weight'])

In [5]:
graph_pyg

Data(x=[174, 100], edge_index=[2, 187], token=[174], label=[187], edge_attr=[187, 1])

In [6]:
edge_index = graph_pyg.edge_index  # shape [2, num_edges]
edge_weights = graph_pyg.edge_attr.squeeze()  # shape [num_edges], if it's single-dim

adj = to_dense_adj(edge_index, edge_attr=edge_weights, max_num_nodes=MAX_NODES)
adj = adj.squeeze(0)

In [7]:
from src.multi_graph.visualization import visualization_l0

visualization_l0(
    nx_graph=graph_nx,
    output_filename="test.html"
)

In [1]:
adj_np = adj.cpu().numpy()[:500, :500]

# 4. Plot heatmap using seaborn
plt.figure(figsize=(17, 13), dpi=300)
sns.heatmap(adj_np, annot=False, cmap="grey", square=True, cbar=True)
plt.title("Adjacency Matrix Heatmap")
plt.xlabel("Node index")
plt.ylabel("Node index")
plt.show()

NameError: name 'adj' is not defined

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dati di esempio: una lista di numeri
data = edge_weights

# Funzione per la SMA (Simple Moving Average Study on Implicit Behavioral Cloning for Multimodal Behavior Learning in Autonomous Driving)
def simple_moving_average(x, window):
    if window <= 0:
        raise ValueError("Window size must be positive")
    if window > len(x):
        raise ValueError("Window size cannot be larger than data length")
    sma = []
    for i in range(len(x) - window + 1):
        window_vals = x[i : i + window]
        sma.append(sum(window_vals) / window)
    # Per allineare la lunghezza con i dati originali, aggiungo NaN all'inizio
    sma = [np.nan] * (window - 1) + sma
    return sma

# Funzione per la EMA (Exponential Moving Average)
def exponential_moving_average(x, alpha):
    """
    alpha: smoothing factor tra 0 e 1.
    """
    ema = [x[0]]  # primo valore = primo dato
    for i in range(1, len(x)):
        ema_val = alpha * x[i] + (1 - alpha) * ema[-1]
        ema.append(ema_val)
    return ema

# Calcola le medie mobili

sma = simple_moving_average(data, 5)
sma2 = simple_moving_average(data, 20)

ema = exponential_moving_average(data, 0.2)
ema2 = exponential_moving_average(data, 0.1)


# Plot
plt.figure(figsize=(25, 10), dpi=300)

plt.plot(sma, label=f'SMA (window={5})', linestyle='--')
plt.plot(sma2, label=f'SMA (window={20})', linestyle='--')
plt.plot(ema, label=f'EMA (alpha={0.2})', linestyle='-.')
plt.plot(ema2, label=f'EMA (alpha={0.1})', linestyle='-.')
plt.plot(data, label='Dati originali', marker='.', color="grey", alpha=0.5)

plt.xlabel('Indice')
plt.ylabel('Valore')
plt.title('Dati originali + SMA + EMA')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(25, 10), dpi=300)
plt.hist(edge_weights)

# Dataset evaluation

In [ ]:
graph_num_nodes = []
graph_num_edges = []
base_adj = torch.zeros(MAX_NODES, MAX_NODES)
isolates = []

for graph_path in tqdm.tqdm(GRAPHS[:1000]):
    doc_name, label, graph_nx = torch.load(graph_path,  weights_only=False)

    graph_num_nodes.append(graph_nx.number_of_nodes())
    graph_num_edges.append(graph_nx.number_of_edges())

    graph_pyg = from_networkx(graph_nx, group_edge_attrs=['weight'])

    edge_index = graph_pyg.edge_index  # shape [2, num_edges]
    edge_weights = graph_pyg.edge_attr.squeeze()  # shape [num_edges], if it's single-dim

    adj = to_dense_adj(edge_index, edge_attr=edge_weights, max_num_nodes=MAX_NODES)
    adj = adj.squeeze(0)
    base_adj += adj

    num_zero_rows = (adj.cpu().numpy() == 0).all().sum()
    isolates.append(num_zero_rows)

    del doc_name, label, graph_nx, adj, edge_index, edge_weights, graph_pyg




In [ ]:
plt.figure(figsize=(25, 10), dpi=300)
plt.hist(graph_num_edges)

In [ ]:
plt.figure(figsize=(25, 10), dpi=300)
plt.hist(graph_num_nodes)

In [ ]:
plt.figure(figsize=(25, 10), dpi=300)
plt.hist(isolates)

In [ ]:
adj_np = base_adj.cpu().numpy()[:100, :100]

# 4. Plot heatmap using seaborn
plt.figure(figsize=(17, 13), dpi=300)
sns.heatmap(adj_np, annot=False, cmap="grey", square=True, cbar=True)
plt.title("Adjacency Matrix Heatmap")
plt.xlabel("Node index")
plt.ylabel("Node index")
plt.show()